# || NEMO Artificial .tiff generator ||
© Konstantinos Andreadis 2024 (PhD in the Roux Lab & Salbreux Lab at UNIGE, Switzerland)

please cite: K.Andreadis _et al._ "NEMO: Mesh-Based Tangential Nematic Field, Defect, and Morphology Analysis in Volumetric Microscopy Data" (2026,_in preparation_)

In [ ]:
# Import custom scripts
from scripts import analysis, datahandler, visuals, simulation
from importlib import reload
from skimage.draw import line_nd

for module in (analysis, datahandler, visuals, simulation):
    reload(module)

# Import custom scripts
import numpy as np
import os, trimesh

In [ ]:
import numpy as np
import tifffile as tif
from scipy.ndimage import map_coordinates, gaussian_filter


def generate_nemo_tetra_bench(l=256, r_inner=65, r_outer=95, dr=1.5, steps=18):
    print(f">> Generating Tetrahedral +1/2 Defects & Polar Asters...")
    vol = np.zeros((l, l, l), dtype=np.float32)
    center = np.array([l // 2, l // 2, l // 2])
    z, y, x = np.indices((l, l, l))

    # Coordinates
    dx, dy, dz = (x - center[0]), (y - center[1]), (z - center[2])
    dist = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2) + 1e-9

    # Normalized coordinates (for vector math)
    nx, ny, nz = dx / dist, dy / dist, dz / dist

    # 1. Define Tetrahedral Vertices (for inner shell)
    s2 = np.sqrt(2)
    s6 = np.sqrt(6)
    pts = [
        np.array([0, 0, 1]),
        np.array([2 * s2 / 3, 0, -1 / 3]),
        np.array([-s2 / 3, s6 / 3, -1 / 3]),
        np.array([-s2 / 3, -s6 / 3, -1 / 3])
    ]

    # 2. Build the Tetrahedral Flow Field
    # We use a combined phase approach
    total_phase = np.zeros_like(dist)
    for p in pts:
        # Vector from vertex to point
        vx, vy, vz = nx - p[0], ny - p[1], nz - p[2]
        # Project onto tangent plane to get a local 2D angle
        # (This is a simplified projection for the benchmark)
        local_phi = np.arctan2(vy, vx)
        total_phase += 0.5 * local_phi

    # Standard Spherical Basis
    phi = np.arctan2(dy, dx)
    theta = np.arccos(np.clip(nz, -1, 1))

    # Phi-hat and Theta-hat
    p_vec = [-np.sin(phi), np.cos(phi), np.zeros_like(phi)]
    t_vec = [np.cos(phi) * np.cos(theta), np.sin(phi) * np.cos(theta), -np.sin(theta)]

    # Inner Flow (Tetrahedral 1/2 defects)
    flow_in = [p_vec[0] * np.cos(total_phase) + t_vec[0] * np.sin(total_phase),
               p_vec[1] * np.cos(total_phase) + t_vec[1] * np.sin(total_phase),
               p_vec[2] * np.cos(total_phase) + t_vec[2] * np.sin(total_phase)]

    # Outer Flow (2 Asters)
    flow_out = t_vec

    # 3. LIC Advection
    seeds = (np.random.uniform(0, 1, (l, l, l)) > 0.992).astype(np.float32)

    def advect(mask, flow, n_steps):
        acc = np.copy(seeds)
        dt = 0.9
        for d in [1, -1]:
            cz, cy, cx = z.astype(float), y.astype(float), x.astype(float)
            for _ in range(n_steps):
                cx += d * dt * flow[0]
                cy += d * dt * flow[1]
                cz += d * dt * flow[2]
                acc += map_coordinates(seeds, [cz, cy, cx], order=1, mode='constant', cval=0)
        return acc

    print(">> Processing inner (Tetrahedral)...")
    inner_mask = (dist >= r_inner - dr) & (dist <= r_inner + dr)
    vol[inner_mask] = advect(inner_mask, flow_in, steps)[inner_mask] * 120
    vol[inner_mask] += 40

    # print(">> Processing outer (Asters)...")
    # outer_mask = (dist >= r_outer - dr) & (dist <= r_outer + dr)
    # vol[outer_mask] = advect(outer_mask, flow_out, steps)[outer_mask] * 120
    # vol[outer_mask] += 40

    vol = gaussian_filter(vol, sigma=0.6)

    tif.imwrite("nemo_tetra_benchmark.tif", vol.astype(np.uint16))
    print(">> Done.")
    return vol


# Run it!
stack = generate_nemo_tetra_bench(l=256, r_inner=70, r_outer=95, steps=20)
visuals.plot_img(img=stack, scale=(1, 1, 1), unit="px", max_proj=True)

In [ ]:
import numpy as np
import tifffile as tif
from scipy.ndimage import map_coordinates, gaussian_filter


def generate_nemo_master_bench(
        l=256,
        do_inner=True,
        do_outer=True,
        r_inner=70,
        r_outer=100,
        dr=1.5,
        steps=40,  # Increased for longer comet tails
        seed_density=0.012
):
    print(f">> Initializing 3D manifold simulation...")
    vol = np.zeros((l, l, l), dtype=np.float32)
    center = np.array([l // 2, l // 2, l // 2])
    z, y, x = np.indices((l, l, l))

    # Cast to float32 immediately to avoid UFuncTypeError
    dx = (x - center[0]).astype(np.float32)
    dy = (y - center[1]).astype(np.float32)
    dz = (z - center[2]).astype(np.float32)

    dist = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2) + 1e-9
    nx, ny, nz = dx / dist, dy / dist, dz / dist

    # Spherical basis
    phi = np.arctan2(dy, dx)
    theta = np.arccos(np.clip(nz, -1, 1))

    # Tangent plane basis: et (theta-hat), ep (phi-hat)
    et = [np.cos(phi) * np.cos(theta), np.sin(phi) * np.cos(theta), -np.sin(theta)]
    ep = [-np.sin(phi), np.cos(phi), np.zeros_like(phi)]

    # 1. GENERATE SEEDS (Fiber start points)
    seeds = (np.random.uniform(0, 1, (l, l, l)) > (1 - seed_density)).astype(np.float32)

    def advect_fibers(flow, mask, n_steps):
        # We only want to advect seeds that are actually on the shell
        active_seeds = seeds * mask
        acc = np.copy(active_seeds)
        dt = 0.8  # Slightly larger step for smoother curves
        for d in [1, -1]:
            cz, cy, cx = z.astype(float), y.astype(float), x.astype(float)
            for _ in range(n_steps):
                cx += d * dt * flow[0]
                cy += d * dt * flow[1]
                cz += d * dt * flow[2]
                acc += map_coordinates(active_seeds, [cz, cy, cx], order=1, mode='constant', cval=0)
        res = np.zeros_like(acc)
        res[mask] = acc[mask]
        return res

    # --- INNER LAYER (The Tennis Ball: 4 x +1/2 Comets) ---
    if do_inner:
        print(">> Calculating Tetrahedral Ground State (Tennis Ball)...")
        # Perfectly equidistant points (Tetrahedron vertices)
        s = 1 / np.sqrt(3)
        pts = [
            np.array([s, s, s]),
            np.array([s, -s, -s]),
            np.array([-s, s, -s]),
            np.array([-s, -s, s])
        ]

        # We use a Q-tensor-like summation approach
        # Summing (cos(alpha), sin(alpha)) where alpha is the angle to the defect
        # results in a +1/2 defect after dividing the final angle by 2.
        v_sum_x = np.zeros_like(dx)
        v_sum_y = np.zeros_like(dx)

        for p in pts:
            # Vector from defect p to point n
            v_vec = [nx - p[0], ny - p[1], nz - p[2]]
            # Project onto local tangent basis (et, ep)
            v_t = v_vec[0] * et[0] + v_vec[1] * et[1] + v_vec[2] * et[2]
            v_p = v_vec[0] * ep[0] + v_vec[1] * ep[1] + v_vec[2] * ep[2]

            # Normalize the projected vector
            v_mag = np.sqrt(v_t ** 2 + v_p ** 2 + 1e-9)
            v_sum_x += v_t / v_mag
            v_sum_y += v_p / v_mag

        # The director angle is HALF the angle of the summed vectors
        inner_angle = 0.5 * np.arctan2(v_sum_y, v_sum_x)

        flow_in = [et[0] * np.cos(inner_angle) + ep[0] * np.sin(inner_angle),
                   et[1] * np.cos(inner_angle) + ep[1] * np.sin(inner_angle),
                   et[2] * np.cos(inner_angle) + ep[2] * np.sin(inner_angle)]

        inner_mask = (dist >= r_inner - dr) & (dist <= r_inner + dr)
        vol += advect_fibers(flow_in, inner_mask, steps) * 180
        vol[inner_mask] += 40

    # --- OUTER LAYER (Asters) ---
    if do_outer:
        print(">> Calculating Outer Asters...")
        flow_out = [et[0], et[1], et[2]]
        outer_mask = (dist >= r_outer - dr) & (dist <= r_outer + dr)
        vol += advect_fibers(flow_out, outer_mask, steps) * 180
        vol[outer_mask] += 40

    vol = gaussian_filter(vol, sigma=0.5)
    print(f">> Benchmark saved successfully.")
    return vol


# Execution
stack = generate_nemo_master_bench(l=256, do_inner=True, do_outer=True, steps=45)
visuals.plot_img(img=stack, scale=(1, 1, 1), unit="px", max_proj=True)

In [ ]:
tif.imwrite("debug.tif", stack.astype(np.uint16))

In [ ]:
visuals.view_img([stack], scale=(1, 1, 1))

# Sphere (with defects)

In [ ]:
mode = "aster"  # "isotropic" "ring"
data_path = f"/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/3_simulated/sphere/data/{mode}"
fig_path = os.path.join(os.path.dirname(os.path.dirname(data_path)), "figures", os.path.basename(data_path))
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
vec_posdir, t1_raw, t2_raw = simulation.spherical_defect(n=80, mode=mode)
visuals.plot_dir_field(directors=vec_posdir, veclength=0.1, title=f"{mode} on sphere",
                       savefig=os.path.join(fig_path, "directors.png"), figsize=(6, 3.5))
datahandler.save_array(t1_raw, "tan_x", header="t1x,t1y,t1z", folderpath=data_path)
datahandler.save_array(t2_raw, "tan_y", header="t2x,t2y,t2z", folderpath=data_path)
datahandler.save_array(vec_posdir, "directors", header="x,y,z,vx,vy,vz", folderpath=data_path)

In [ ]:
def sphere_half_defects(sphere_radius=50):
    sphere_mesh = trimesh.creation.icosphere(subdivisions=4, radius=sphere_radius)
    sphere_pos = sphere_mesh.vertices
    print(len(sphere_pos))
    t1t2 = analysis.create_tangential_basis(sphere_mesh.vertex_normals)
    dirs = simulation.random_tangential(*t1t2, seed=1)
    dir_field = np.column_stack((sphere_pos, dirs))
    for _ in range(30):
        s, n = analysis.avg_tan_nem_tens(*t1t2, dir_field,
                                         analysis.coord_search_neighbours(sphere_pos, k=len(sphere_pos) // 2))
        dir_field[:, 3:] = n[-1]
    s, n = analysis.avg_tan_nem_tens(*t1t2, dir_field, analysis.coord_search_neighbours(sphere_pos, k=20))
    dir_field[:, 3:] = n[-1]
    visuals.plot_dir_field(dir_field, veclength=10, veccolor=s[-1], manual_vminmax=[0, 1])
    print(len(dir_field))
    return dir_field, s[-1]


dir_field, s = sphere_half_defects()
visuals.view_3d_vector_field(dir_field[:, :3], dir_field[:, 3:], visuals.color_scalar(s, manual_vminmax=[0, 1]),
                             length=10)


def draw_streamlines_volume(dir_field, steps=20, step_size=0.3):
    positions = dir_field[:, :3]
    vectors = dir_field[:, 3:]
    L = int(1.2 * np.ptp(positions, axis=0).mean())
    volume = np.zeros((L, L, L), dtype=np.float32)
    center = positions.mean(axis=0)
    norm_positions = ((positions - center) + L / 2).astype(float)
    for pos, vec in zip(norm_positions, vectors):
        for _ in range(steps):
            next_pos = pos + step_size * vec
            if np.any(next_pos < 0) or np.any(next_pos >= L):
                break
            rr = line_nd(pos, next_pos, endpoint=True)
            rr = [np.clip(r, 0, L - 1) for r in rr]
            volume[tuple(rr)] = 1
            pos = next_pos
    volume = analysis.gaussian_blur(volume, sigma=0.1, renorm=False)
    volume = (volume / volume.max() * 255).astype(np.uint8)
    return volume


vol = draw_streamlines_volume(dir_field)
visuals.plot_img(vol, scale=(1, 1, 1), unit="px")
visuals.plot_img(vol, scale=(1, 1, 1), unit="px", max_proj=True)

visuals.view_img(img_list=[vol], scale=(1, 1, 1))

rootpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/"
datahandler.create_dir(rootpath)
datahandler.save_tiff(vol, os.path.join(rootpath, "debug_half_defects.tiff"))

# 3D Hedgehog

In [ ]:
def fibonacci_sphere(n_points):
    """Return approximately uniform directions over a sphere."""
    i = np.arange(n_points)
    phi = np.arccos(1 - 2 * (i + 0.5) / n_points)  # polar angle
    theta = np.pi * (1 + 5 ** 0.5) * (i + 0.5)  # golden angle (azimuth)
    return phi, theta


n_rays = 100  # Adjust for density
Nz, Ny, Nx = 200, 200, 200
scale = (1, 1, 1)
test = np.zeros((Nz, Ny, Nx))

# Center of the star
center = np.array([Nz // 2, Ny // 2, Nx // 2])

# Get uniformly distributed directions
phi, theta = fibonacci_sphere(n_rays)

# Ray directions
num_steps = 300
r = np.linspace(0, min(Nz, Ny, Nx), num_steps)

dz = np.outer(np.cos(phi), r)
dy = np.outer(np.sin(phi) * np.sin(theta), r)
dx = np.outer(np.sin(phi) * np.cos(theta), r)

# Add to center
z = dz + center[0]
y = dy + center[1]
x = dx + center[2]

# Round and index
z_idx = np.round(z).astype(int).ravel()
y_idx = np.round(y).astype(int).ravel()
x_idx = np.round(x).astype(int).ravel()

# Mask in-bounds
valid = (
        (z_idx >= 0) & (z_idx < Nz) &
        (y_idx >= 0) & (y_idx < Ny) &
        (x_idx >= 0) & (x_idx < Nx)
)

# Use boolean mask to prevent accumulation
mask = np.zeros_like(test, dtype=bool)
mask[z_idx[valid], y_idx[valid], x_idx[valid]] = True
test[mask] = 1
datahandler.save_tiff(test, "/Users/andreadi/Downloads/3dstar.tiff")
# visuals.view_img([test], scale=(1, 1, 1))

# Surface (with defect)

In [ ]:
mode_list = ["flat", "parabolic_up", "parabolic_down", "saddle"]
m_d_list = [-1, -0.5, 0, 0.5, 1, 1.5, 2, "isotropic", "+1  ring", "+1  vortex"]
mode_list = ["flat"]
m_d_list = [-1]
L = 0.2
for mode in mode_list:
    for m_d in m_d_list:
        print(f"======= Mode: {mode}, m_d: {m_d}")
        data_path = f"/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/3_simulated/{mode}-surface/data/{m_d}-defect"
        fig_path = os.path.join(os.path.dirname(os.path.dirname(data_path)), "figures", os.path.basename(data_path))
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        N = simulation.odd_number_box(L)
        X, Y = np.meshgrid(np.linspace(-L, L, N), np.linspace(-L, L, N))
        Z, dz_dx, dz_dy = simulation.generate_surface(mode=mode, x_=X, y_=Y)
        if type(m_d) == str:
            if m_d == "isotropic":
                x, y, u, v = simulation.isotropic_2d_field(l=L, n=N)
            else:
                x, y, u, v = simulation.point_defect_2d(l=L, n=N, defect_type=m_d)
        else:
            x, y, u, v = simulation.point_defect_2d_charge(l=L, n=N, defect_charge=m_d)
        visuals.plot_vector_field(x, y, u, v, defect=m_d, figsize=(3, 3),
                                  savefigpath=os.path.join(fig_path, "defect.png"))
        vec_posdir, normals = simulation.surface_warp(X, Y, Z, dz_dx, dz_dy, u, v)
        t1_raw, t2_raw = analysis.create_tangential_basis(normals)
        visuals.plot_dir_field(directors=vec_posdir, t1_raw=None, t2_raw=None, normals=normals, veclength=0.03,
                               view_init=(30, 30), figsize=(5, 3.5), title=f"{m_d} defect on {mode} surface",
                               savefig=os.path.join(fig_path, "directors.png"))
        datahandler.save_array(t1_raw, "tan_x", header="t1x,t1y,t1z", folderpath=data_path)
        datahandler.save_array(t2_raw, "tan_y", header="t2x,t2y,t2z", folderpath=data_path)
        datahandler.save_array(vec_posdir, "directors", header="x,y,z,vx,vy,vz", folderpath=data_path)

# Mesh (.ply) Loading

In [ ]:
mesh_folder_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Model/Meshes"
# mesh_name = "tetrahedon.ply"
mesh_name = "cylinder.ply"
mesh_name = "sphere.ply"
mesh_path = os.path.join(mesh_folder_path, mesh_name)
data_path = f"/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/3_simulated/mesh/data/{mesh_name[:-4]}"
fig_path = os.path.join(os.path.dirname(os.path.dirname(data_path)), "figures", os.path.basename(data_path))
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
mesh = trimesh.load(mesh_path)
t1_raw, t2_raw = analysis.create_tangential_basis(mesh.vertex_normals)
vec_posdir = np.empty(shape=(len(mesh.vertices), 6))
vec_posdir[:, :3] = mesh.vertices
vec_posdir[:, 3:] = t1_raw * np.random.uniform(-1, 1, size=(len(t1_raw), 1)) + t2_raw * np.random.uniform(-1, 1, size=(
    len(t2_raw), 1))
visuals.plot_dir_field(directors=vec_posdir, t1_raw=None, t2_raw=None, veclength=0.1, view_init=(45, 45),
                       savefig=os.path.join(fig_path, "directors.png"), figsize=(5, 3))
datahandler.save_array(t1_raw, "tan_x", header="t1x,t1y,t1z", folderpath=data_path)
datahandler.save_array(t2_raw, "tan_y", header="t2x,t2y,t2z", folderpath=data_path)
datahandler.save_array(vec_posdir, "directors", header="x,y,z,vx,vy,vz", folderpath=data_path)

# Multi-Layer Spherical Shell Actin

In [ ]:
L = 200
shell = simulation.spherical_shell(L, r=int(L / 3), dr=4, sigma=0.0, num_rings=21, shell_intensity=1,
                                   theta_width=0.02, phi_width=0.02)
# rootpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/3_simulated/shell"
rootpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/"
datahandler.create_dir(rootpath)
shell_save_path = os.path.join(rootpath, "debug.tiff")
datahandler.save_tiff(shell, shell_save_path)
visuals.plot_img(img=shell, scale=(1, 1, 1), unit="px", figsize=(10, 2))
visuals.view_img(img_list=[shell], color_list=["green"], title_list=["Generated"], scale=(1, 1, 1))

In [ ]:
L = 200
shell = simulation.uniform_shell(L, r=int(L * 0.36), dr=6, sigma=0)
rootpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/3_simulated/control_shell"
if not os.path.exists(rootpath):
    os.makedirs(rootpath)
shell_save_path = os.path.join(rootpath, "shell.tiff")
datahandler.save_tiff(shell, shell_save_path)
# visuals.view_img(img_list=[shell], color_list=["green"], title_list=["Generated"], scale=(1, 1, 1))
visuals.plot_img(img=shell, scale=(1, 1, 1), unit="px", figsize=(10, 2))

# Bulk Fake Fibres

In [ ]:
def generate_spherical_grid(num_points):
    """
    Generate uniformly spaced points on a sphere using the Fibonacci sphere method.
    """
    indices = np.arange(0, num_points, dtype=float) + 0.5
    phi = 2 * np.pi * indices / num_points  # Azimuthal angle
    theta = np.arccos(1 - 2 * indices / num_points)  # Polar angle
    return theta, phi


def spherical_to_cartesian(theta, phi):
    """
    Convert spherical coordinates to Cartesian coordinates.
    """
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return np.array([x, y, z])


def generate_collagen_fiber(start_point, orientation, length, volume, thickness=0):
    """
    Generate a fiber from a starting point in a specific direction, with optional thickness.
    """
    points = []
    for t in np.linspace(0, length, int(length)):
        point = start_point + t * orientation
        if all(0 <= point[i] < volume.shape[i] for i in range(3)):
            for dx in range(-thickness, thickness + 1):
                for dy in range(-thickness, thickness + 1):
                    for dz in range(-thickness, thickness + 1):
                        neighbor = point + np.array([dx, dy, dz])
                        if all(0 <= neighbor[i] < volume.shape[i] for i in range(3)):
                            points.append(tuple(map(int, neighbor)))
    return points


def generate_uniform_hedgehog(volume_size, num_fibers, fiber_length, thickness=0):
    """
    Generate a uniformly spaced radial hedgehog pattern of fibers emanating from the center of the volume.
    """
    img = np.zeros((volume_size, volume_size, volume_size))
    center_point = np.array([volume_size // 2] * 3)

    # Generate uniform points on the sphere
    theta, phi = generate_spherical_grid(num_fibers)
    orientations = [spherical_to_cartesian(theta[i], phi[i]) for i in range(num_fibers)]

    for orientation in orientations:
        fiber_points = generate_collagen_fiber(center_point, orientation, fiber_length, img, thickness)
        for p in fiber_points:
            img[p] = 1

    return img


# Parameters
L = 200
num_fibers = 40
fiber_length = L // 2  # Half the volume size to ensure fibers stay within bounds
thickness = 0  # Adjust thickness as needed

# Generate the 3D aster of fibers
fibres = generate_uniform_hedgehog(L, num_fibers, fiber_length, thickness)

fibres = analysis.gaussian_blur(fibres, sigma=1, renorm=False)
rootpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/3_simulated/fibres"
if not os.path.exists(rootpath):
    os.makedirs(rootpath)
datahandler.save_tiff(fibres, os.path.join(rootpath, "fibres.tiff"))
visuals.plot_img(img=fibres, scale=(1, 1, 1), unit="px", figsize=(10, 2))
visuals.plot_img(img=fibres, scale=(1, 1, 1), unit="px", figsize=(10, 2), max_proj=True)
visuals.view_img(img_list=[fibres], color_list=["green"], title_list=["Generated"], scale=(1, 1, 1))

# Trimesh Mesh Library of Shapes

In [ ]:
radius = 10
mesh = trimesh.creation.icosphere(radius=radius, subdivisions=5)
mesh = trimesh.creation.uv_sphere(radius=radius)
# mesh = trimesh.creation.capsule(radius=radius, height=radius)
# mesh = trimesh.creation.cone(radius=radius, height=radius)
# mesh = trimesh.creation.cylinder(radius=radius, height=3 * radius)

# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Model/Meshes/tetrahedon.ply"
# path = "/Users/andreadi/Downloads/untitled.ply"

# 2D Defect Gallery

In [ ]:
# savefigpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Code/defect_gallery"
# visualise = False
# if visualise:
#     L_ = [23]
# else:
#     L_ = [10, 20, 50, 100, 200, 500]
# charge_range = True
# if charge_range:
#     if visualise:
#         defect_types = [1]
#     else:
#         defect_types = np.arange(-2, 2.25, 0.25)
#     defect_charge = defect_types
# else:
#     defect_types = ["-1", "-0.5", "0", "+0.5", "+1  aster", "+1  ring", "+1  vortex", "+1.5", "+2"]
#     defect_charge = [round(float(i[:4]), 1) for i in defect_types]
# div_ = np.empty(shape=(len(L_), len(defect_types)))
# curl_ = np.empty(shape=(len(L_), len(defect_types)))
# charge_ = np.empty(shape=(len(L_), len(defect_types)))
# for i, L in enumerate(L_):
#     for j, defect in enumerate(defect_types):
#         if charge_range:
#             X, Y, U, V = simulation.point_defect_2d_charge(l=L, n=21, defect_charge=defect)
#         else:
#             X, Y, U, V = simulation.point_defect_2d(l=L, n=21, defect_type=defect)
#         divergence, curl = analysis.compute_divergence_and_curl(X, Y, U, V)
#         div_[i, j] = np.round(np.sum(divergence), 3)
#         curl_[i, j] = np.round(np.sum(curl), 3)
#         charge_[i, j] = defect_charge[j]
#         if visualise:
#             visuals.plot_vector_field_with_div_curl(X, Y, U, V, divergence, curl, defect,
#                                                     savefigpath=os.path.join(savefigpath,
#                                                                              f"{defect}_L-{L}_div_curl.png"),
#                                                     figsize=(10, 3))
#             visuals.plot_vector_field_styles(X, Y, U, V, defect,
#                                              savefigpath=os.path.join(savefigpath, f"{defect}_L-{L}.png"),
#                                              figsize=(10, 3))
#
# visuals.plot_curl_div_vs_charge(l_=L_, charge_=charge_, div_=div_, curl_=curl_,
#                                 savefigpath=os.path.join(savefigpath, "div_and_curl.png"), figsize=(8, 7))